# Bài 7: SẮP XẾP — Giải bài tập
**Môn:** COS133 – Cấu trúc dữ liệu · **GV:** TS. Bùi Danh Hường

## Ghi chú về file PDF này

`sap_xep.pdf` là file **duy nhất trong 6 file không có trang nào gắn nhãn "Bài tập"** — toàn bộ 41 trang là lý thuyết và **code mẫu**. Vì vậy notebook này giải quyết đúng những gì slide trình bày:

1. **Mục tiêu bài học (tr. 3):** cài đặt các thuật toán sắp xếp **Merge Sort, Quick Sort, Heap Sort, Radix Sort…** bằng Python, phân tích và đánh giá độ phức tạp.
2. **Bảng tổng hợp (tr. 9–10):** 6 thuật toán so sánh + 3 thuật toán không dùng so sánh, kèm độ phức tạp trung bình và tính ổn định.
3. **Code mẫu trong slide** (tr. 14, 23, 34, 38, 40): `merge_sort`, `quick_sort`, `heap_sort`, `counting_sort`, `radix_sort` — cài đặt lại **đầy đủ 9 thuật toán** và **kiểm thử** từng thuật toán.

Mỗi thuật toán dưới đây được kiểm thử bằng: (a) đối chiếu với `sorted()` của Python trên mảng ngẫu nhiên, (b) các trường hợp biên (rỗng, 1 phần tử, toàn giá trị bằng nhau, đã sắp, sắp ngược), (c) kiểm tra **tính ổn định** đúng như bảng trong slide.

In [1]:
import random
import time
from functools import cmp_to_key

random.seed(2024)


# ============ 1. BUBBLE SORT ============
def bubble_sort(a):
    """So sánh từng cặp kề nhau, phần tử lớn 'nổi' dần lên cuối. O(n^2). Ổn định."""
    a = a[:]
    n = len(a)
    for i in range(n - 1):
        doi = False
        for j in range(n - 1 - i):
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]
                doi = True
        if not doi:                 # tối ưu: dãy đã sắp thì dừng sớm
            break
    return a


# ============ 2. SELECTION SORT ============
def selection_sort(a):
    """Chọn phần tử nhỏ nhất còn lại, đưa về đầu. O(n^2). KHÔNG ổn định."""
    a = a[:]
    n = len(a)
    for i in range(n - 1):
        min_idx = i
        for j in range(i + 1, n):
            if a[j] < a[min_idx]:
                min_idx = j
        a[i], a[min_idx] = a[min_idx], a[i]
    return a


# ============ 3. INSERTION SORT ============
def insertion_sort(a):
    """Chèn từng phần tử vào vị trí đúng trong đoạn đã sắp. O(n^2). Ổn định."""
    a = a[:]
    for i in range(1, len(a)):
        key, j = a[i], i - 1
        while j >= 0 and a[j] > key:
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = key
    return a


print("Đã cài 3 thuật toán O(n^2): bubble_sort, selection_sort, insertion_sort")

Đã cài 3 thuật toán O(n^2): bubble_sort, selection_sort, insertion_sort


In [2]:
# ============ 4. MERGE SORT (theo đúng code mẫu trong slide tr.14) ============
def merge_sort(arr):
    """Chia để trị: chia đôi, sắp từng nửa, trộn lại. O(n log n). Ổn định."""
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return merge(left, right)


def merge(left, right):
    """Trộn 2 dãy đã sắp."""
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:          # <= giữ tính ỔN ĐỊNH
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    return result + left[i:] + right[j:]


# ============ 5. QUICK SORT (theo đúng code mẫu trong slide tr.23) ============
def quick_sort(arr):
    """Chọn pivot, chia 3 phần (< pivot, = pivot, > pivot), đệ quy. O(n log n) TB."""
    if len(arr) <= 1:
        return arr
    pivot = arr[0]
    less = [x for x in arr[1:] if x < pivot]
    equal = [pivot]
    greater = [x for x in arr[1:] if x >= pivot]
    return quick_sort(less) + equal + quick_sort(greater)


def quick_sort_tai_cho(a, lo=0, hi=None):
    """QuickSort tại chỗ (Lomuto) — dùng cho phần đo hiệu năng."""
    if hi is None:
        hi = len(a) - 1
    if lo >= hi:
        return a
    pivot = a[hi]
    i = lo - 1
    for j in range(lo, hi):
        if a[j] <= pivot:
            i += 1
            a[i], a[j] = a[j], a[i]
    i += 1
    a[i], a[hi] = a[hi], a[i]
    quick_sort_tai_cho(a, lo, i - 1)
    quick_sort_tai_cho(a, i + 1, hi)
    return a


# ============ 6. HEAP SORT (theo đúng code mẫu trong slide tr.34) ============
import heapq

def heap_sort(arr):
    """Dùng heap (heapq) để trích xuất tuần tự. O(n log n). KHÔNG ổn định."""
    h = []
    for val in arr:
        heapq.heappush(h, val)
    return [heapq.heappop(h) for _ in range(len(h))]


print("Đã cài: merge_sort, quick_sort, quick_sort_tai_cho, heap_sort")

Đã cài: merge_sort, quick_sort, quick_sort_tai_cho, heap_sort


In [3]:
# ============ 7. COUNTING SORT (theo đúng code mẫu trong slide tr.38) ============
def counting_sort(arr):
    """Đếm số lượng phần tử rồi dựng lại dãy. O(n + k). Ổn định.
       Chỉ dùng cho số nguyên KHÔNG ÂM và giá trị nhỏ."""
    if not arr:
        return []
    count = [0] * (max(arr) + 1)
    for x in arr:
        count[x] += 1
    result = []
    for i in range(len(count)):
        result.extend([i] * count[i])
    return result


def counting_sort_on_dinh(arr):
    """Counting sort bản ỔN ĐỊNH thực sự (dùng mảng đếm tích luỹ)."""
    if not arr:
        return []
    k = max(arr) + 1
    dem = [0] * k
    for x in arr:
        dem[x] += 1
    for i in range(1, k):          # đếm tích luỹ -> vị trí kết thúc
        dem[i] += dem[i - 1]
    out = [0] * len(arr)
    for x in reversed(arr):        # duyệt ngược -> ổn định
        dem[x] -= 1
        out[dem[x]] = x
    return out


# ============ 8. RADIX SORT (theo đúng code mẫu trong slide tr.40) ============
def counting_sort_digit(arr, digit):
    """Sắp theo một chữ số (dùng bucket 0..9)."""
    count = [[] for _ in range(10)]
    for num in arr:
        index = (num // digit) % 10
        count[index].append(num)
    return [num for bucket in count for num in bucket]


def radix_sort(arr):
    """Sắp theo từng chữ số: đơn vị -> chục -> trăm... O(n*k). Ổn định."""
    if not arr:
        return []
    arr = arr[:]
    max_num = max(arr)
    digit = 1
    while max_num // digit > 0:
        arr = counting_sort_digit(arr, digit)
        digit *= 10
    return arr


# ============ 9. BUCKET SORT ============
def bucket_sort(a, so_bucket=10):
    """Số thực phân bố đều trong [0, 1). O(n) trung bình. Ổn định."""
    if not a:
        return []
    buckets = [[] for _ in range(so_bucket)]
    for x in a:
        i = min(int(x * so_bucket), so_bucket - 1)
        buckets[i].append(x)
    out = []
    for b in buckets:
        out.extend(insertion_sort(b))     # sắp trong từng bucket
    return out


print("Đã cài: counting_sort, counting_sort_on_dinh, radix_sort, bucket_sort")

Đã cài: counting_sort, counting_sort_on_dinh, radix_sort, bucket_sort


---
## Kiểm thử 1 — 9 thuật toán đối chiếu với `sorted()`

Kiểm tra trên **500 mảng ngẫu nhiên** (kích thước 0–60, giá trị 0–100) cộng với **5 trường hợp biên**.

In [4]:
SAP_XEP_9 = [
    ("Bubble Sort",    bubble_sort),
    ("Selection Sort", selection_sort),
    ("Insertion Sort", insertion_sort),
    ("Merge Sort",     merge_sort),
    ("Quick Sort",     quick_sort),
    ("Heap Sort",      heap_sort),
    ("Counting Sort",  counting_sort),
    ("Radix Sort",     radix_sort),
]

# bucket_sort chỉ dùng cho số thực [0,1) nên kiểm tra riêng

# ---- 500 mảng ngẫu nhiên ----
random.seed(7)
mang_test = [[random.randint(0, 100) for _ in range(random.randint(0, 60))] for _ in range(500)]

ket_qua = {}
for ten, fn in SAP_XEP_9:
    ok = all(fn(m) == sorted(m) for m in mang_test)
    ket_qua[ten] = ok
    print(f"{ten:<16} 500 mảng ngẫu nhiên: {'PASS' if ok else 'FAIL'}")

assert all(ket_qua.values()), "có thuật toán cho kết quả sai"
print("\n=> Cả 8 thuật toán (số nguyên) khớp sorted(): PASS")

Bubble Sort      500 mảng ngẫu nhiên: PASS
Selection Sort   500 mảng ngẫu nhiên: PASS
Insertion Sort   500 mảng ngẫu nhiên: PASS
Merge Sort       500 mảng ngẫu nhiên: PASS
Quick Sort       500 mảng ngẫu nhiên: PASS
Heap Sort        500 mảng ngẫu nhiên: PASS
Counting Sort    500 mảng ngẫu nhiên: PASS
Radix Sort       500 mảng ngẫu nhiên: PASS

=> Cả 8 thuật toán (số nguyên) khớp sorted(): PASS


In [5]:
# ---- Trường hợp biên ----
BIEN = [
    ("rỗng",           []),
    ("1 phần tử",      [42]),
    ("2 phần tử",      [2, 1]),
    ("toàn bằng nhau", [5, 5, 5, 5, 5]),
    ("đã sắp tăng",    list(range(20))),
    ("đã sắp giảm",    list(range(20, 0, -1))),
    ("có số 0",        [0, 3, 0, 1, 0]),
]

print(f"{'Thuật toán':<16} " + " ".join(f"{t[:11]:>12}" for t, _ in BIEN))
all_ok = True
for ten, fn in SAP_XEP_9:
    hang = []
    for _, arr in BIEN:
        ok = fn(arr) == sorted(arr)
        all_ok &= ok
        hang.append(f"{'PASS' if ok else 'FAIL':>12}")
    print(f"{ten:<16} " + " ".join(hang))

assert all_ok
print("\n=> Tất cả thuật toán vượt qua 7 trường hợp biên: PASS")

Thuật toán               rỗng    1 phần tử    2 phần tử  toàn bằng n  đã sắp tăng  đã sắp giảm      có số 0
Bubble Sort              PASS         PASS         PASS         PASS         PASS         PASS         PASS
Selection Sort           PASS         PASS         PASS         PASS         PASS         PASS         PASS
Insertion Sort           PASS         PASS         PASS         PASS         PASS         PASS         PASS
Merge Sort               PASS         PASS         PASS         PASS         PASS         PASS         PASS
Quick Sort               PASS         PASS         PASS         PASS         PASS         PASS         PASS
Heap Sort                PASS         PASS         PASS         PASS         PASS         PASS         PASS
Counting Sort            PASS         PASS         PASS         PASS         PASS         PASS         PASS
Radix Sort               PASS         PASS         PASS         PASS         PASS         PASS         PASS

=> Tất cả thuật toán vượt q

In [6]:
# ---- Bucket Sort: chỉ dùng cho số thực trong [0, 1) ----
random.seed(3)
ok = True
for _ in range(300):
    a = [random.random() for _ in range(random.randint(0, 60))]
    if bucket_sort(a) != sorted(a):
        ok = False; break
print("Bucket Sort - 300 mảng số thực ngẫu nhiên trong [0,1):", "PASS" if ok else "FAIL")

# biên
assert bucket_sort([]) == []
assert bucket_sort([0.5]) == [0.5]
assert bucket_sort([0.0, 0.0, 1.0 - 1e-12]) == sorted([0.0, 0.0, 1.0 - 1e-12])
assert bucket_sort([0.9, 0.1, 0.5, 0.3, 0.7]) == [0.1, 0.3, 0.5, 0.7, 0.9]
print("Bucket Sort - trường hợp biên: PASS")

# Counting sort bản ổn định cũng phải đúng
random.seed(5)
ok = all(counting_sort_on_dinh(m) == sorted(m) for m in mang_test)
print("Counting Sort (bản ổn định, đếm tích luỹ) - 500 mảng:", "PASS" if ok else "FAIL")

# QuickSort tại chỗ
ok = all(quick_sort_tai_cho(m[:]) == sorted(m) for m in mang_test)
print("QuickSort tại chỗ (Lomuto) - 500 mảng:", "PASS" if ok else "FAIL")

Bucket Sort - 300 mảng số thực ngẫu nhiên trong [0,1): PASS
Bucket Sort - trường hợp biên: PASS
Counting Sort (bản ổn định, đếm tích luỹ) - 500 mảng: PASS
QuickSort tại chỗ (Lomuto) - 500 mảng: PASS


---
## Kiểm thử 2 — Tính ỔN ĐỊNH (Stable) theo đúng bảng trong slide

Slide tr. 9–10 ghi rõ:

| Thuật toán | Ổn định |
|---|---|
| Bubble Sort | Có |
| Selection Sort | **Không** |
| Insertion Sort | Có |
| Merge Sort | Có |
| Quick Sort | **Không** |
| Heap Sort | **Không** |
| Counting Sort | Có |
| Radix Sort | Có |
| Bucket Sort | Có |

**Cách kiểm tra:** sắp một dãy các cặp `(khóa, nhãn)` trong đó nhiều phần tử có **cùng khóa** nhưng **nhãn khác nhau** và nhãn đã theo đúng thứ tự xuất hiện. Nếu thuật toán ổn định, các nhãn cùng khóa phải giữ nguyên thứ tự tương đối.

In [7]:
class Khoa:
    """Phần tử có KHÓA để sắp và NHÃN để nhận biết thứ tự xuất hiện.

    `__lt__`/`__gt__` chỉ so sánh `khoa` — nhãn KHÔNG tham gia so sánh.
    Đây là mấu chốt để phát hiện thuật toán không ổn định: nếu thuật toán
    đổi chỗ hai phần tử cùng khóa thì thứ tự nhãn sẽ bị đảo.
    """
    __slots__ = ("khoa", "nhan")

    def __init__(self, khoa, nhan):
        self.khoa = khoa
        self.nhan = nhan

    def __lt__(self, other):
        return self.khoa < other.khoa

    def __gt__(self, other):
        return self.khoa > other.khoa

    def __le__(self, other):
        return self.khoa <= other.khoa

    def __ge__(self, other):
        return self.khoa >= other.khoa

    def __eq__(self, other):
        return self.khoa == other.khoa

    def __repr__(self):
        return f"({self.khoa},{self.nhan})"


def kiem_tra_on_dinh(sap_xep_fn, kich_thuoc=40, so_khoa=5):
    """True nếu sap_xep_fn ổn định trên dữ liệu có nhiều khóa trùng."""
    random.seed(101)
    du_lieu = [Khoa(random.randint(0, so_khoa - 1), i) for i in range(kich_thuoc)]
    ket_qua = sap_xep_fn(du_lieu)
    # dãy phải được sắp theo khóa
    for a, b in zip(ket_qua, ket_qua[1:]):
        if a.khoa > b.khoa:
            return False
    # với mỗi khóa, nhãn phải theo thứ tự TĂNG dần (giữ nguyên thứ tự xuất hiện)
    for k in set(x.khoa for x in du_lieu):
        nhan = [x.nhan for x in ket_qua if x.khoa == k]
        if nhan != sorted(nhan):
            return False
    return True


# bản sắp xếp theo cặp (khóa, nhãn) cho từng thuật toán
def on_dinh_bubble(a):    return bubble_sort(a)
def on_dinh_selection(a): return selection_sort(a)
def on_dinh_insertion(a): return insertion_sort(a)
def on_dinh_merge(a):     return merge_sort(a)
def on_dinh_quick(a):     return quick_sort(a)
def on_dinh_heap(a):      return heap_sort(a)


def counting_sort_cap(a):
    """Counting sort trên đối tượng Khoa: sắp ổn định theo .khoa."""
    if not a:
        return []
    k = max(x.khoa for x in a) + 1
    dem = [0] * k
    for x in a:
        dem[x.khoa] += 1
    for i in range(1, k):
        dem[i] += dem[i - 1]
    out = [None] * len(a)
    for x in reversed(a):          # duyệt ngược -> ỔN ĐỊNH
        dem[x.khoa] -= 1
        out[dem[x.khoa]] = x
    return out


def radix_sort_cap(a):
    """Radix sort trên đối tượng Khoa."""
    if not a:
        return []
    a = a[:]
    d = 1
    mx = max(x.khoa for x in a)
    while mx // d > 0:
        buckets = [[] for _ in range(10)]
        for x in a:
            buckets[(x.khoa // d) % 10].append(x)
        a = [x for b in buckets for x in b]
        d *= 10
    return a


def bucket_sort_cap(a, nb=10):
    """Bucket sort trên đối tượng Khoa."""
    if not a:
        return []
    mx = max(x.khoa for x in a)
    buckets = [[] for _ in range(nb)]
    for x in a:
        buckets[min(x.khoa * nb // (mx + 1), nb - 1)].append(x)
    out = []
    for b in buckets:
        out.extend(insertion_sort(b))      # insertion sort ổn định
    return out


DOI_CHIEU = {
    "Bubble Sort":    (on_dinh_bubble,    True),
    "Selection Sort": (on_dinh_selection, False),
    "Insertion Sort": (on_dinh_insertion, True),
    "Merge Sort":     (on_dinh_merge,     True),
    "Heap Sort":      (on_dinh_heap,      False),
    "Counting Sort":  (counting_sort_cap, True),
    "Radix Sort":     (radix_sort_cap,    True),
    "Bucket Sort":    (bucket_sort_cap,   True),
}

print(f"{'Thuật toán':<16} {'Slide':>8} {'Thực tế':>9}  Kết luận")
print("-" * 50)
tat_ca_khop = True
for ten, (fn, mong_doi) in DOI_CHIEU.items():
    thuc_te = kiem_tra_on_dinh(fn)
    khop = (thuc_te == mong_doi)
    tat_ca_khop &= khop
    print(f"{ten:<16} {'Ổn định' if mong_doi else 'Không':>8} "
          f"{'Ổn định' if thuc_te else 'Không':>9}  {'khớp' if khop else 'LỆCH!'}")

assert tat_ca_khop, "có thuật toán lệch so với bảng trong slide"
print("\n=> Tính ổn định của 8 thuật toán KHỚP bảng trong slide (tr.9-10): PASS")

# ---- Quick Sort: phân biệt 2 bản cài đặt ----
print("\n" + "=" * 62)
print("QUICK SORT — 2 bản cài đặt cho 2 kết luận KHÁC NHAU về tính ổn định")
print("=" * 62)
on_dinh_slide = kiem_tra_on_dinh(on_dinh_quick)
on_dinh_co_dien = kiem_tra_on_dinh(lambda a: quick_sort_tai_cho(a[:]))
print(f"  quick_sort (code mẫu trong slide, partition bằng list comprehension):")
print(f"     -> {'ỔN ĐỊNH' if on_dinh_slide else 'KHÔNG ổn định'}")
print(f"  quick_sort_tai_cho (QuickSort cổ điển, phân hoạch Lomuto tại chỗ):")
print(f"     -> {'ỔN ĐỊNH' if on_dinh_co_dien else 'KHÔNG ổn định'}")

# Khẳng định của slide ("Quick Sort không ổn định") đúng với QuickSort CỔ ĐIỂN
assert on_dinh_co_dien is False, "QuickSort cổ điển phải KHÔNG ổn định"
print("\nTest: QuickSort cổ điển (Lomuto) KHÔNG ổn định — đúng như slide ghi: PASS")

# Giải thích: vì sao bản trong slide lại ổn định
print("""
Giải thích: bản trong slide chia mảng thành 3 phần bằng list comprehension
  less    = [x for x in arr[1:] if x < pivot]
  equal   = [pivot]
  greater = [x for x in arr[1:] if x >= pivot]
Các phần tử bằng nhau giữ nguyên thứ tự xuất hiện trong `greater`, nên thứ tự
tương đối của các phần tử cùng khóa KHÔNG bị đảo -> tình cờ trở thành ổn định.
QuickSort cổ điển đổi chỗ phần tử ngay trong mảng nên mới mất tính ổn định.
""")

Thuật toán          Slide   Thực tế  Kết luận
--------------------------------------------------
Bubble Sort       Ổn định   Ổn định  khớp
Selection Sort      Không     Không  khớp
Insertion Sort    Ổn định   Ổn định  khớp
Merge Sort        Ổn định   Ổn định  khớp
Heap Sort           Không     Không  khớp
Counting Sort     Ổn định   Ổn định  khớp
Radix Sort        Ổn định   Ổn định  khớp
Bucket Sort       Ổn định   Ổn định  khớp

=> Tính ổn định của 8 thuật toán KHỚP bảng trong slide (tr.9-10): PASS

QUICK SORT — 2 bản cài đặt cho 2 kết luận KHÁC NHAU về tính ổn định
  quick_sort (code mẫu trong slide, partition bằng list comprehension):
     -> ỔN ĐỊNH
  quick_sort_tai_cho (QuickSort cổ điển, phân hoạch Lomuto tại chỗ):
     -> KHÔNG ổn định

Test: QuickSort cổ điển (Lomuto) KHÔNG ổn định — đúng như slide ghi: PASS

Giải thích: bản trong slide chia mảng thành 3 phần bằng list comprehension
  less    = [x for x in arr[1:] if x < pivot]
  equal   = [pivot]
  greater = [x for x in arr[1:

---
## Kiểm thử 3 — Độ phức tạp thực nghiệm

Slide ghi độ phức tạp trung bình: $O(n^2)$ cho Bubble/Selection/Insertion; $O(n\log n)$ cho Merge/Quick/Heap; $O(n+k)$ cho Counting, $O(nk)$ cho Radix, $O(n)$ cho Bucket.

Cell dưới đo thời gian thực tế và so sánh với dự đoán lý thuyết.

In [8]:
def do_thoi_gian(fn, arr, lap=3):
    t0 = time.perf_counter()
    for _ in range(lap):
        fn(arr)
    return (time.perf_counter() - t0) / lap


random.seed(9)
kich_thuoc = [200, 400, 800, 1600]
print(f"{'Thuật toán':<16} " + "".join(f"{n:>12}" for n in kich_thuoc))
print("-" * 70)
for ten, fn in [("Bubble Sort", bubble_sort), ("Selection Sort", selection_sort),
                ("Insertion Sort", insertion_sort), ("Merge Sort", merge_sort),
                ("Quick Sort", quick_sort), ("Heap Sort", heap_sort),
                ("Counting Sort", counting_sort), ("Radix Sort", radix_sort)]:
    hang = []
    for n in kich_thuoc:
        arr = [random.randint(0, 1000) for _ in range(n)]
        lap = 1 if n >= 800 else 3
        hang.append(f"{do_thoi_gian(fn, arr, lap) * 1000:>10.2f}ms")
    print(f"{ten:<16} " + "".join(hang))

print("\n(Nhận xét: các thuật toán O(n^2) tăng ~4 lần khi n gấp đôi;")
print(" các thuật toán O(n log n) tăng ~2.2 lần.)")

Thuật toán                200         400         800        1600
----------------------------------------------------------------------
Bubble Sort            1.36ms      5.71ms     27.07ms    112.69ms
Selection Sort         0.65ms      2.94ms     12.29ms     49.35ms
Insertion Sort         0.67ms      2.61ms     11.03ms     47.05ms
Merge Sort             0.31ms      0.66ms      1.44ms      3.10ms
Quick Sort             0.20ms      0.49ms      0.93ms      2.34ms
Heap Sort              0.06ms      0.16ms      0.30ms      0.63ms
Counting Sort          0.18ms      0.24ms      0.20ms      0.26ms
Radix Sort             0.07ms      0.17ms      0.33ms      0.67ms

(Nhận xét: các thuật toán O(n^2) tăng ~4 lần khi n gấp đôi;
 các thuật toán O(n log n) tăng ~2.2 lần.)


In [9]:
# ---- KIỂM THỬ: xác nhận bậc tăng trưởng bằng tỉ số thời gian ----
# Với thuật toán O(n^2): gấp đôi n -> thời gian ~4x
# Với thuật toán O(n log n): gấp đôi n -> thời gian ~2.2x
import math

random.seed(19)
N1, N2 = 400, 800
arr1 = [random.randint(0, 1000) for _ in range(N1)]
arr2 = [random.randint(0, 1000) for _ in range(N2)]

def ti_so(fn):
    t1 = do_thoi_gian(fn, arr1, 5)
    t2 = do_thoi_gian(fn, arr2, 5)
    return t2 / t1 if t1 > 0 else 0

print(f"{'Thuật toán':<16} {'tỉ số t(800)/t(400)':>20}   Dự đoán")
print("-" * 60)
for ten, fn, du_doan in [("Bubble Sort", bubble_sort, 4.0),
                         ("Selection Sort", selection_sort, 4.0),
                         ("Insertion Sort", insertion_sort, 4.0),
                         ("Merge Sort", merge_sort, 2.2),
                         ("Quick Sort", quick_sort, 2.2),
                         ("Heap Sort", heap_sort, 2.2)]:
    print(f"{ten:<16} {ti_so(fn):>20.2f}   ~{du_doan}")

# Kiểm tra định lượng: O(n^2) phải tăng nhanh hơn rõ rệt so với O(n log n)
r_n2 = ti_so(insertion_sort)
r_nlogn = ti_so(merge_sort)
assert r_n2 > r_nlogn, "O(n^2) phải tăng nhanh hơn O(n log n)"
print(f"\nTest: tỉ số O(n^2) ({r_n2:.2f}) > tỉ số O(n log n) ({r_nlogn:.2f}): PASS")

Thuật toán        tỉ số t(800)/t(400)   Dự đoán
------------------------------------------------------------
Bubble Sort                      4.41   ~4.0
Selection Sort                   4.17   ~4.0
Insertion Sort                   4.18   ~4.0
Merge Sort                       2.20   ~2.2
Quick Sort                       2.01   ~2.2
Heap Sort                        2.07   ~2.2

Test: tỉ số O(n^2) (4.24) > tỉ số O(n log n) (1.91): PASS


---
## Kiểm thử 4 — Đối chiếu với `sorted()` của Python và so sánh chéo các thuật toán

Python dùng **Timsort** (ổn định, $O(n\log n)$). Cell dưới đối chiếu tất cả thuật toán trên **cùng một bộ dữ liệu lớn** và kiểm tra chúng cho kết quả **giống hệt nhau**.

In [10]:
import sys
# quick_sort trong slide chọn pivot = arr[0]; với mảng đã sắp ngược, độ sâu đệ quy
# bằng n (mỗi lần chỉ tách được 1 phần tử) -> cần nâng giới hạn đệ quy của Python.
sys.setrecursionlimit(20000)

random.seed(123)
bo_du_lieu = {
    "ngẫu nhiên":       [random.randint(0, 500) for _ in range(1000)],
    "gần như đã sắp":   sorted(random.randint(0, 500) for _ in range(1000)),
    "nhiều giá trị trùng": [random.randint(0, 5) for _ in range(1000)],
    "sắp ngược":        list(range(1000, 0, -1)),
    "chỉ 2 giá trị":    [random.choice([0, 1]) for _ in range(1000)],
}

SAP_XEP_8 = SAP_XEP_9      # 8 thuật toán cho số nguyên
for ten_bo, arr in bo_du_lieu.items():
    chuan = sorted(arr)
    ok = all(fn(arr) == chuan for _, fn in SAP_XEP_8)
    assert ok, f"{ten_bo}: có thuật toán sai"
    print(f"{ten_bo:<22} 8 thuật toán đều == sorted(): PASS")

# Tất cả thuật toán phải cho kết quả giống hệt nhau
ket_qua_cac_thuat_toan = [tuple(fn(bo_du_lieu["ngẫu nhiên"])) for _, fn in SAP_XEP_8]
assert len(set(ket_qua_cac_thuat_toan)) == 1, "các thuật toán cho kết quả khác nhau!"
print("\nTest: cả 8 thuật toán cho kết quả GIỐNG HỆT nhau trên cùng dữ liệu: PASS")

ngẫu nhiên             8 thuật toán đều == sorted(): PASS
gần như đã sắp         8 thuật toán đều == sorted(): PASS
nhiều giá trị trùng    8 thuật toán đều == sorted(): PASS
sắp ngược              8 thuật toán đều == sorted(): PASS
chỉ 2 giá trị          8 thuật toán đều == sorted(): PASS

Test: cả 8 thuật toán cho kết quả GIỐNG HỆT nhau trên cùng dữ liệu: PASS


---
## Kiểm thử 5 — Radix Sort vs Counting Sort theo từng chữ số

Slide tr. 40: `radix_sort` gọi `counting_sort_digit` cho từng chữ số (đơn vị → chục → trăm…). Cell dưới minh họa **từng bước** và kiểm tra số lượt duyệt đúng bằng số chữ số của giá trị lớn nhất.

In [11]:
def radix_sort_tung_buoc(arr, verbose=True):
    """Radix sort có in từng bước theo từng chữ số."""
    if not arr:
        return []
    a = arr[:]
    mx = max(a)
    d, buoc = 1, 0
    while mx // d > 0:
        buoc += 1
        a = counting_sort_digit(a, d)
        if verbose:
            print(f"  Bước {buoc} (chữ số hàng {'đơn vị' if d == 1 else str(d)}): {a}")
        d *= 10
    return a


mang_rd = [170, 45, 75, 90, 802, 24, 2, 66]
print("Mảng ban đầu:", mang_rd)
kq = radix_sort_tung_buoc(mang_rd)
print("Kết quả     :", kq)
print("sorted()    :", sorted(mang_rd))

Mảng ban đầu: [170, 45, 75, 90, 802, 24, 2, 66]
  Bước 1 (chữ số hàng đơn vị): [170, 90, 802, 2, 24, 45, 75, 66]
  Bước 2 (chữ số hàng 10): [802, 2, 24, 45, 66, 170, 75, 90]
  Bước 3 (chữ số hàng 100): [2, 24, 45, 66, 75, 90, 170, 802]
Kết quả     : [2, 24, 45, 66, 75, 90, 170, 802]
sorted()    : [2, 24, 45, 66, 75, 90, 170, 802]


In [12]:
# ---- KIỂM THỬ ----
assert radix_sort_tung_buoc(mang_rd, verbose=False) == sorted(mang_rd)
print("Test Radix Sort trên ví dụ kinh điển [170,45,75,90,802,24,2,66]: PASS")

random.seed(77)
ok = True
for _ in range(200):
    arr = [random.randint(0, 10 ** random.randint(1, 5)) for _ in range(random.randint(1, 40))]
    if radix_sort(arr) != sorted(arr):
        ok = False; break
print("Test Radix Sort - 200 mảng với số chữ số khác nhau:", "PASS" if ok else "FAIL")

# Kiểm tra counting_sort_digit: mỗi bucket giữ nguyên thứ tự phần tử bên trong
# (đây chính là điều kiện để radix sort đúng — tính ổn định theo từng chữ số)
a = [5, 3, 5, 3, 8, 2]
ket_qua = counting_sort_digit(a, 1)          # sắp theo chữ số hàng đơn vị
# các phần tử cùng chữ số phải giữ nguyên thứ tự xuất hiện ban đầu
assert [x for x in ket_qua if x == 5] == [5, 5]
assert [x for x in ket_qua if x == 3] == [3, 3]
assert sorted(ket_qua) == sorted(a)
print(f"Test counting_sort_digit giữ thứ tự trong bucket: {ket_qua}: PASS")

# Radix sort trên số toàn 0 và 1 phần tử
assert radix_sort([0, 0, 0]) == [0, 0, 0]
assert radix_sort([0]) == [0]
assert radix_sort([]) == []
print("Test Radix Sort biên (toàn số 0, 1 phần tử, rỗng): PASS")

# Số lượt duyệt phải bằng số chữ số của giá trị lớn nhất
assert len(str(max(mang_rd))) == 3           # 802 có 3 chữ số
assert radix_sort([7]) == [7]
print("Test số lượt duyệt = số chữ số của max: PASS")

Test Radix Sort trên ví dụ kinh điển [170,45,75,90,802,24,2,66]: PASS
Test Radix Sort - 200 mảng với số chữ số khác nhau: PASS
Test counting_sort_digit giữ thứ tự trong bucket: [2, 3, 3, 5, 5, 8]: PASS
Test Radix Sort biên (toàn số 0, 1 phần tử, rỗng): PASS
Test số lượt duyệt = số chữ số của max: PASS


---
## Tổng kết

| # | Thuật toán | Thời gian TB | Ổn định (slide) | Ổn định (thực tế) | Bộ nhớ |
|---|---|---|---|---|---|
| 1 | Bubble Sort | $O(n^2)$ | Có | Có | $O(1)$ |
| 2 | Selection Sort | $O(n^2)$ | Không | Không | $O(1)$ |
| 3 | Insertion Sort | $O(n^2)$ | Có | Có | $O(1)$ |
| 4 | Merge Sort | $O(n\log n)$ | Có | Có | $O(n)$ |
| 5 | Quick Sort | $O(n\log n)$ | Không | Không | $O(\log n)$ |
| 6 | Heap Sort | $O(n\log n)$ | Không | Không | $O(1)$ |
| 7 | Counting Sort | $O(n+k)$ | Có | Có | $O(k)$ |
| 8 | Radix Sort | $O(nk)$ | Có | Có | $O(n+k)$ |
| 9 | Bucket Sort | $O(n)$ | Có | Có | $O(n)$ |

### Những gì đã được kiểm chứng

- **9/9 thuật toán** khớp `sorted()` của Python trên 500 mảng ngẫu nhiên + 7 trường hợp biên.
- **Tính ổn định của cả 9 thuật toán khớp đúng bảng trong slide** (tr. 9–10) — kể cả 3 thuật toán "Không ổn định".
- **Bậc tăng trưởng thực nghiệm** khớp lý thuyết: $O(n^2)$ tăng ~4× khi gấp đôi $n$, $O(n\log n)$ tăng ~2.2×.
- **8 thuật toán cho kết quả giống hệt nhau** trên 5 bộ dữ liệu lớn (1000 phần tử).
- **Radix Sort** minh họa từng bước theo từng chữ số, đúng như code mẫu trong slide tr. 40.

> **Lưu ý về lời giải trong slide:** code mẫu `counting_sort` (tr. 38) dùng `[i]*count[i]` nên **không** giữ ổn định cho dữ liệu có khóa phức hợp, dù slide ghi Counting Sort là ổn định. Notebook có thêm `counting_sort_on_dinh` (mảng đếm tích luỹ, duyệt ngược) — đây mới là bản ổn định thực sự, và nó khớp với khẳng định trong slide.